# CSCI 345 · Week 11 Lab: Recurrent Neural Networks from Scratch

**Goal:** turn the ideas from the lecture into working code. In this lab you will:

| Part | What you build | Lecture idea |
|---|---|---|
| 1 | A bag-of-words "model" that can't tell two sentences apart | Why order matters |
| 2 | One RNN step, then a full forward pass, in NumPy | $h_t = \tanh(W_{hh}h_{t-1} + W_{xh}x_t)$ |
| 3 | Many-to-one and many-to-many outputs from the same RNN | RNN patterns |
| 4 | Experiments on how memory fades and how gradients vanish | Long-term dependencies |
| 5 | Training an RNN with backpropagation through time (BPTT) | BPTT |
| 6 | A character-level RNN that generates text | One-to-many, feeding outputs back in |
| 7 | *(Colab)* The same ideas in PyTorch, plus RNN vs. LSTM | Fixing vanishing gradients |
| 8 | Practice challenges | Going further |

**How to use this notebook**
- Run cells top to bottom (`Shift + Enter`).
- Cells marked **✏️ TODO** need your code. Replace `raise NotImplementedError` with your solution.
- Most TODOs are followed by a **✅ check** cell. If it prints `✅`, move on. If it raises an error, read the message.
- 🧠 **Think** boxes are questions to answer in a sentence or two (in a markdown cell or on paper).

**Where to run it:** Parts 1–6 need only NumPy and Matplotlib (any Jupyter install, or Google Colab). Part 7 needs PyTorch, so run it on **Google Colab** (`File → Upload notebook`).

Estimated time: about 2 hours for Parts 1–6. Part 7 and the challenges are for extra practice.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(345)   # fixed seed so everyone gets the same numbers
print("NumPy", np.__version__, "ready")

---
## Part 1 · Why order matters (10 min)

In lecture we saw two sentences with **the same words in a different order**:

- *"Market falls into bear territory"* → topic: **Trading**
- *"Bear falls into market territory"* → topic: **Unknown**

A *bag-of-words* representation just counts how often each word appears. Let's see what it does with these two sentences.

In [ ]:
s1 = "market falls into bear territory"
s2 = "bear falls into market territory"

vocab = sorted(set(s1.split()) | set(s2.split()))
print("Vocabulary:", vocab)

def bag_of_words(sentence, vocab):
    # Return a vector of word counts, one entry per word in vocab.
    # ✏️ YOUR CODE HERE (hint: count how many times each vocab word appears in sentence.split())
    raise NotImplementedError("Complete this TODO")

v1, v2 = bag_of_words(s1, vocab), bag_of_words(s2, vocab)
print("s1 ->", v1)
print("s2 ->", v2)

In [ ]:
# ✅ check
assert v1.shape == (len(vocab),), "Your vector should have one entry per vocabulary word."
assert np.array_equal(v1, v2), "Hmm, the two count vectors should be identical."
print("✅ The two sentences have IDENTICAL bag-of-words vectors.")
print("   Any model that only sees these vectors must give them the same prediction.")

> 🧠 **Think:** A feed-forward net that takes one word per input slot *would* see the order. Name two problems with that approach that we discussed in lecture. *(Hint: sentence length, and learning the word "bear" separately in every position.)*

---
## Part 2 · The RNN forward pass from scratch (25 min)

### 2a. One step with a 1-number hidden state

The vanilla RNN update is

$$h_t = \tanh(w_{hh}\, h_{t-1} + w_{xh}\, x_t)$$

Start with **scalars** so you can check the arithmetic by hand. This is the worked example from the lecture: $w_{hh}=0.5$, $w_{xh}=1.0$, $h_0=0$, inputs $x=(1, 0, 0)$.

In [ ]:
def rnn_step_scalar(h_prev, x, w_hh=0.5, w_xh=1.0):
    # ✏️ TODO: return the new hidden state h_t
    # ✏️ YOUR CODE HERE (hint: one line using np.tanh)
    raise NotImplementedError("Complete this TODO")

h = 0.0
for t, x in enumerate([1, 0, 0], start=1):
    h = rnn_step_scalar(h, x)
    print(f"step {t}: x = {x}  ->  h = {h:.3f}")

In [ ]:
# ✅ check against the lecture's worked example
h, hs = 0.0, []
for x in [1, 0, 0]:
    h = rnn_step_scalar(h, x)
    hs.append(round(float(h), 3))
assert hs == [0.762, 0.363, 0.180] or np.allclose(hs, [0.762, 0.363, 0.180], atol=1e-3), f"Got {hs}"
print("✅ Matches the slide: 0.762, 0.363, 0.180")

### 2b. The real thing: vectors and weight matrices

Now the hidden state is a vector of size $H$, the input a vector of size $D$, and we add an output:

$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h) \qquad\qquad y_t = W_{hy} h_t + b_y$$

| Symbol | Shape | Meaning |
|---|---|---|
| $x_t$ | $(D,)$ | input at step $t$ |
| $h_t$ | $(H,)$ | hidden state ("memory") |
| $W_{xh}$ | $(H, D)$ | input → hidden |
| $W_{hh}$ | $(H, H)$ | previous hidden → hidden |
| $W_{hy}$ | $(C, H)$ | hidden → output (C outputs) |

**Key idea from lecture:** the same weights are reused at every step. Notice that `rnn_forward` below loops over time but never creates new weights.

In [ ]:
class VanillaRNN:
    def __init__(self, input_size, hidden_size, output_size, seed=0):
        r = np.random.default_rng(seed)
        self.Wxh = r.normal(0, 0.1, (hidden_size, input_size))
        self.Whh = r.normal(0, 0.1, (hidden_size, hidden_size))
        self.bh  = np.zeros(hidden_size)
        self.Why = r.normal(0, 0.1, (output_size, hidden_size))
        self.by  = np.zeros(output_size)
        self.H = hidden_size

    def step(self, h_prev, x):
        # ✏️ TODO: one time step. Return the new hidden state (shape (H,)).
        # ✏️ YOUR CODE HERE (hint: use self.Whh, self.Wxh, self.bh and np.tanh)
        raise NotImplementedError("Complete this TODO")

    def output(self, h):
        return self.Why @ h + self.by

    def forward(self, xs, h0=None):
        # xs: array of shape (T, D). Returns all hidden states, shape (T, H).
        h = np.zeros(self.H) if h0 is None else h0
        hs = []
        # ✏️ TODO: loop over time, update h with self.step, and collect every h
        # ✏️ YOUR CODE HERE (hint: for x in xs: ... hs.append(h))
        raise NotImplementedError("Complete this TODO")
        return np.array(hs)

    def num_params(self):
        return sum(w.size for w in [self.Wxh, self.Whh, self.bh, self.Why, self.by])

In [ ]:
# ✅ check
rnn = VanillaRNN(input_size=3, hidden_size=4, output_size=2)
xs = rng.normal(size=(5, 3))          # a sequence of T=5 inputs, each of size D=3
hs = rnn.forward(xs)
assert hs.shape == (5, 4), f"Expected (5, 4), got {hs.shape}"
assert np.allclose(hs[0], np.tanh(rnn.Wxh @ xs[0])), "h_1 should be tanh(Wxh x_1) when h_0 = 0 and b_h = 0"
assert np.allclose(hs[1], np.tanh(rnn.Whh @ hs[0] + rnn.Wxh @ xs[1])), "h_2 should use h_1"
print("✅ forward pass works. Hidden states:\n", hs)

In [ ]:
# Same weights, any length: the parameter count does NOT depend on T
for T in [3, 30, 300]:
    hs = rnn.forward(rng.normal(size=(T, 3)))
    print(f"T = {T:>3}: hidden states {hs.shape}, parameters = {rnn.num_params()}")

> 🧠 **Think:** Why does the parameter count stay the same when $T$ grows from 3 to 300? How is that different from a feed-forward net that takes all $T$ inputs at once?

---
## Part 3 · Same RNN, different patterns (10 min)

The lecture showed one-to-many, many-to-one and many-to-many. They all use the **same recurrent core**. The only difference is *which hidden states you read outputs from* (and what you feed in).

In [ ]:
def many_to_one(rnn, xs):
    # One prediction for the whole sequence (e.g., sentiment).
    # ✏️ TODO: run the RNN and return the output computed from the LAST hidden state
    # ✏️ YOUR CODE HERE (hint: rnn.forward(xs)[-1], then rnn.output(...))
    raise NotImplementedError("Complete this TODO")

def many_to_many(rnn, xs):
    # One prediction per time step (e.g., tagging each word).
    # ✏️ TODO: return an array of shape (T, C), one output per hidden state
    # ✏️ YOUR CODE HERE (hint: apply rnn.output to every hidden state)
    raise NotImplementedError("Complete this TODO")

xs = rng.normal(size=(6, 3))
print("many-to-one  output shape:", many_to_one(rnn, xs).shape)
print("many-to-many output shape:", many_to_many(rnn, xs).shape)

In [ ]:
# ✅ check
assert many_to_one(rnn, xs).shape == (2,)
assert many_to_many(rnn, xs).shape == (6, 2)
assert np.allclose(many_to_many(rnn, xs)[-1], many_to_one(rnn, xs)), "The last many-to-many output should equal the many-to-one output"
print("✅ Both patterns work, and they share the same final output.")

> 🧠 **Think:** For each task, which function would you use? (a) classifying a tweet as spam/not spam, (b) predicting the part of speech of every word, (c) image captioning. Which one needs something we haven't written yet?

---
## Part 4 · How long does an RNN remember? (20 min)

### 4a. Memory fades in the forward pass

Feed a single "pulse" ($x_1 = 1$, then all zeros) into the scalar RNN and watch $h_t$. We'll try several values of the recurrent weight $w_{hh}$.

In [ ]:
def pulse_response(w_hh, T=20):
    h, out = 0.0, []
    for t in range(T):
        x = 1.0 if t == 0 else 0.0
        h = rnn_step_scalar(h, x, w_hh=w_hh)
        out.append(h)
    return np.array(out)

plt.figure(figsize=(8, 4))
for w in [0.5, 0.9, 1.0, 1.5]:
    plt.plot(range(1, 21), pulse_response(w), marker="o", ms=3, label=f"w_hh = {w}")
plt.xlabel("time step t"); plt.ylabel("hidden state h_t")
plt.title("How long does the pulse at t = 1 stay in memory?")
plt.legend(); plt.grid(alpha=.3); plt.show()

> 🧠 **Think:** With $w_{hh}=0.5$, after how many steps is the memory below 0.01? What happens with $w_{hh}=1.5$, and why doesn't it blow up to infinity? *(Hint: what does tanh do to large numbers?)*

### 4b. Gradients vanish (or explode) in the backward pass

During BPTT, the gradient flowing from step $T$ back to step $1$ is multiplied at every step by

$$\frac{\partial h_t}{\partial h_{t-1}} = \text{diag}(1 - h_t^2)\, W_{hh}$$

So the gradient that reaches the start is (roughly) a **product of $T$ such matrices**. Let's measure its size for random $W_{hh}$ at different scales.

In [ ]:
def gradient_norm_through_time(scale, T=50, H=32, seed=0):
    # Returns ||d h_T / d h_k|| (as an operator norm) for k = T, T-1, ..., 1
    r = np.random.default_rng(seed)
    Whh = r.normal(0, scale / np.sqrt(H), (H, H))   # 'scale' roughly sets how much Whh stretches vectors
    Wxh = r.normal(0, 1.0, (H, 1))
    h, hs = np.zeros(H), []
    for t in range(T):
        h = np.tanh(Whh @ h + Wxh @ r.normal(size=1))
        hs.append(h)
    J = np.eye(H)                 # running product of Jacobians
    norms = []
    for t in reversed(range(T)):
        norms.append(np.linalg.norm(J, 2))
        # ✏️ TODO: multiply J by the Jacobian dh_t/dh_{t-1} = diag(1 - h_t^2) @ Whh
        # ✏️ YOUR CODE HERE (hint: J = J @ (np.diag(1 - hs[t]**2) @ Whh))
        raise NotImplementedError("Complete this TODO")
    return np.array(norms)      # norms[k] = size of the gradient after going back k steps

plt.figure(figsize=(8, 4))
for scale in [0.5, 1.0, 2.0, 4.0]:
    plt.semilogy(gradient_norm_through_time(scale), label=f"W_hh scale = {scale}")
plt.xlabel("number of steps back in time"); plt.ylabel("gradient size (log scale)")
plt.title("Vanishing (and exploding) gradients in BPTT")
plt.legend(); plt.grid(alpha=.3, which="both"); plt.tight_layout(); plt.show()

In [ ]:
# ✅ check
n = gradient_norm_through_time(0.5)
assert n[0] == 1.0 and n[-1] < 1e-6, "With a small W_hh, the gradient should shrink to almost nothing."
print(f"✅ With scale 0.5, after 49 steps back the gradient is about {n[-1]:.1e}. It has vanished.")

> 🧠 **Think:** Which curves vanish and which explode? Connect this to the slide's examples $0.5^{10} \approx 0.001$ and $1.5^{10} \approx 58$. Which problem does **gradient clipping** fix, and why can't clipping fix the other one?

---
## Part 5 · Training an RNN with BPTT (30 min)

Now let's train a **many-to-one** RNN on a task that *requires* long-term memory:

> **The memory task.** The input is a random sequence of bits, e.g. `1 0 0 1 1 0 1 …`. The label is **the first bit**. To get it right, the RNN must carry that one bit through every later step.

This is a tiny version of *"I grew up in France … I speak fluent ___"*: the clue is at the very beginning.

We'll process a **batch** of $B$ sequences at once, so shapes are `x[t]: (D, B)` and `h[t]: (H, B)`.

In [ ]:
def make_memory_batch(B, T, rng):
    # x: (T, 1, B) random bits; y: (B,) = the first bit of each sequence
    x = rng.integers(0, 2, size=(T, 1, B)).astype(float)
    y = x[0, 0].astype(int)
    return x, y

x, y = make_memory_batch(B=4, T=8, rng=rng)
print("sequences (one per row):\n", x[:, 0, :].T.astype(int))
print("labels (first bit):", y)

### 5a. Forward + backward (BPTT)

Read the forward pass first. It's exactly Part 2, plus a softmax/cross-entropy loss on the **last** hidden state.

The backward pass walks **backward through time**. At each step:

1. `dh` is the gradient arriving at $h_t$ (from the loss, or from step $t+1$).
2. Go back through the tanh: `draw = (1 - h_t²) * dh`.
3. **Add** this step's contribution to each shared weight's gradient (lecture: *"sum the gradients over all time steps"*).
4. Pass the gradient on to $h_{t-1}$: `dh = W_hhᵀ @ draw`.

In [ ]:
def init_params(H, D=1, C=2, seed=0):
    r = np.random.default_rng(seed)
    return {"Wxh": r.normal(0, 0.5, (H, D)),
            "Whh": r.normal(0, 1 / np.sqrt(H), (H, H)),
            "bh":  np.zeros((H, 1)),
            "Why": r.normal(0, 0.1, (C, H)),
            "by":  np.zeros((C, 1))}

def forward_backward(P, x, y):
    T, D, B = x.shape
    H = P["Whh"].shape[0]

    # ---------- forward ----------
    hs = {-1: np.zeros((H, B))}
    for t in range(T):
        hs[t] = np.tanh(P["Wxh"] @ x[t] + P["Whh"] @ hs[t - 1] + P["bh"])
    logits = P["Why"] @ hs[T - 1] + P["by"]                      # many-to-one: last state only
    logits = logits - logits.max(axis=0)                          # numerical stability
    probs = np.exp(logits) / np.exp(logits).sum(axis=0)
    loss = -np.log(probs[y, np.arange(B)] + 1e-12).mean()
    acc = (probs.argmax(axis=0) == y).mean()

    # ---------- backward (BPTT) ----------
    G = {k: np.zeros_like(v) for k, v in P.items()}
    dlogits = probs.copy()
    dlogits[y, np.arange(B)] -= 1
    dlogits /= B
    G["Why"] = dlogits @ hs[T - 1].T
    G["by"] = dlogits.sum(axis=1, keepdims=True)

    dh = P["Why"].T @ dlogits          # gradient arriving at h_T
    grad_norms = []                     # we'll record ||dh|| at every step
    for t in reversed(range(T)):
        grad_norms.append(np.linalg.norm(dh) / B)
        # ✏️ TODO (5 lines): backprop through tanh, accumulate G["bh"], G["Wxh"], G["Whh"], then update dh
        # ✏️ YOUR CODE HERE (hint: draw = (1 - hs[t]**2) * dh ; G[...] += ... ; dh = P["Whh"].T @ draw)
        raise NotImplementedError("Complete this TODO")
    return loss, acc, G, grad_norms[::-1]

### 5b. Gradient check: never trust a hand-written backward pass

Compare your analytic gradients with a **numerical** estimate:
$\dfrac{\partial L}{\partial w} \approx \dfrac{L(w+\epsilon) - L(w-\epsilon)}{2\epsilon}$. If BPTT is right, the relative error should be tiny (< 1e-5).

In [ ]:
def grad_check(P, x, y, eps=1e-5, n_checks=5, seed=0):
    r = np.random.default_rng(seed)
    _, _, G, _ = forward_backward(P, x, y)
    worst = 0.0
    for name in ["Wxh", "Whh", "bh", "Why"]:
        for _ in range(n_checks):
            idx = tuple(r.integers(0, s) for s in P[name].shape)
            old = P[name][idx]
            P[name][idx] = old + eps; lp = forward_backward(P, x, y)[0]
            P[name][idx] = old - eps; lm = forward_backward(P, x, y)[0]
            P[name][idx] = old
            num, ana = (lp - lm) / (2 * eps), G[name][idx]
            rel = abs(num - ana) / max(1e-8, abs(num) + abs(ana))
            worst = max(worst, rel)
        print(f"{name:>4}: max relative error so far = {worst:.2e}")
    return worst

P = init_params(H=8)
x, y = make_memory_batch(B=16, T=6, rng=rng)
err = grad_check(P, x, y)
assert err < 1e-5, "❌ Gradients don't match. Re-check the 5 lines in the BPTT loop."
print("✅ BPTT gradients match the numerical gradients.")

### 5c. Train it

We use **Adagrad** (a simple adaptive learning rate) and **gradient clipping**, the fix for exploding gradients from lecture.

In [ ]:
def train_memory_task(T, H=16, steps=2000, lr=0.05, batch=64, seed=0, verbose=True):
    r = np.random.default_rng(seed)
    P = init_params(H, seed=seed)
    cache = {k: np.zeros_like(v) for k, v in P.items()}
    history = []
    for step in range(steps):
        x, y = make_memory_batch(batch, T, r)
        loss, acc, G, _ = forward_backward(P, x, y)
        for k in P:
            np.clip(G[k], -5, 5, out=G[k])                       # gradient clipping
            cache[k] += G[k] ** 2
            P[k] -= lr * G[k] / np.sqrt(cache[k] + 1e-8)          # Adagrad update
        history.append(loss)
        if verbose and step % 500 == 0:
            print(f"step {step:>4}  loss {loss:.3f}  batch acc {acc:.2f}")
    x, y = make_memory_batch(1000, T, r)
    _, test_acc, _, grad_norms = forward_backward(P, x, y)
    return P, test_acc, history, grad_norms

P, acc, hist, _ = train_memory_task(T=5)
print(f"\nTest accuracy with T = 5: {acc:.1%}")

### 5d. Experiment: make the sequence longer

Same model, same training budget. Only the distance between the clue (the first bit) and the prediction (the end) changes.

In [ ]:
lengths = [5, 10, 20, 30, 40]
results = {}
for T in lengths:
    P, acc, hist, norms = train_memory_task(T=T, verbose=False)
    results[T] = (acc, hist, norms)
    print(f"T = {T:>2}:  test accuracy = {acc:.1%}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar([str(T) for T in lengths], [results[T][0] for T in lengths], color="#6C548E")
ax[0].axhline(0.5, ls="--", c="gray"); ax[0].text(0, 0.52, "chance", color="gray")
ax[0].set_xlabel("sequence length T"); ax[0].set_ylabel("test accuracy"); ax[0].set_title("Can the RNN remember the first bit?")
for T in [5, 20, 40]:
    ax[1].semilogy(range(1, T + 1), results[T][2], label=f"T = {T}")
ax[1].set_xlabel("time step t (1 = where the clue is)"); ax[1].set_ylabel("gradient size at h_t")
ax[1].set_title("Gradient reaching each time step (after training)"); ax[1].legend()
plt.tight_layout(); plt.show()

> 🧠 **Think:**
> 1. At what length does the vanilla RNN fail? What accuracy does it get, and what does that number mean?
> 2. In the right-hand plot, look at how much each curve **drops from its right end (the output) to its left end (the clue)**. Ignore the overall height; once the loss is near zero, all gradients are small. How many orders of magnitude does the gradient lose for $T = 40$? Connect this to the "vanishing gradients" slide.
> 3. Try `train_memory_task(T=40, H=64)` or a larger `steps`. Does it help much? *(We'll fix this properly with an LSTM in Part 7.)*

---
## Part 6 · Generating text with a character-level RNN (30 min)

This is the **one-to-many / feed-the-output-back-in** pattern from the lecture, and the same idea as the Colab text-generation demo.

- **Training:** many-to-many. At every position, predict the *next character*.
- **Generating:** start from a seed character, sample the next one, feed it back in, and repeat.

Our training text is the opening of *Alice's Adventures in Wonderland* (public domain). You can swap in any text you like.

In [ ]:
text = """Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, 'and what is the use of a book,' thought Alice 'without pictures or conversations?'
So she was considering in her own mind (as well as she could, for the hot day made her feel very sleepy and stupid), whether the pleasure of making a daisy-chain would be worth the trouble of getting up and picking the daisies, when suddenly a White Rabbit with pink eyes ran close by her.
There was nothing so very remarkable in that; nor did Alice think it so very much out of the way to hear the Rabbit say to itself, 'Oh dear! Oh dear! I shall be late!' (when she thought it over afterwards, it occurred to her that she ought to have wondered at this, but at the time it all seemed quite natural); but when the Rabbit actually took a watch out of its waistcoat-pocket, and looked at it, and then hurried on, Alice started to her feet, for it flashed across her mind that she had never before seen a rabbit with either a waistcoat-pocket, or a watch to take out of it, and burning with curiosity, she ran across the field after it, and fortunately was just in time to see it pop down a large rabbit-hole under the hedge.
In another moment down went Alice after it, never once considering how in the world she was to get out again.
The rabbit-hole went straight on like a tunnel for some way, and then dipped suddenly down, so suddenly that Alice had not a moment to think about stopping herself before she found herself falling down a very deep well."""

chars = sorted(set(text))
V = len(chars)
char_to_ix = {c: i for i, c in enumerate(chars)}
ix_to_char = {i: c for i, c in enumerate(chars)}
data = [char_to_ix[c] for c in text]
print(f"{len(text)} characters, vocabulary size V = {V}")
print("vocabulary:", "".join(chars))

### 6a. One-hot encoding

Each character becomes a vector of length $V$ with a single 1, just like the one-hot word vectors in the lecture's named-entity example.

In [ ]:
def one_hot(ix, V):
    # ✏️ TODO: return a column vector of shape (V, 1) with a 1 at position ix
    # ✏️ YOUR CODE HERE (hint: np.zeros((V, 1)) then set one entry)
    raise NotImplementedError("Complete this TODO")

# ✅ check
v = one_hot(char_to_ix["A"], V)
assert v.shape == (V, 1) and v.sum() == 1 and v[char_to_ix["A"], 0] == 1
print("✅ one_hot works")

### 6b. The model: forward and BPTT over a chunk of text

This is the classic *min-char-rnn* design: we train on chunks of `seq_len` characters and carry the hidden state from one chunk to the next. The structure is the same as Part 5, except now **there is a loss at every time step** (many-to-many), so `dh` gets a contribution from the output at each step *plus* the gradient coming back from the future. We train it with **Adam**, the optimizer you'll use most often in practice.

In [ ]:
class CharRNN:
    def __init__(self, V, H=100, seed=1):
        r = np.random.default_rng(seed)
        self.V, self.H = V, H
        self.P = {"Wxh": r.normal(0, 0.01, (H, V)),
                  "Whh": r.normal(0, 0.01, (H, H)),
                  "Why": r.normal(0, 0.01, (V, H)),
                  "bh": np.zeros((H, 1)), "by": np.zeros((V, 1))}
        self.m = {k: np.zeros_like(v) for k, v in self.P.items()}   # Adam state
        self.v = {k: np.zeros_like(v) for k, v in self.P.items()}
        self.t = 0

    def loss_and_grads(self, inputs, targets, h_prev):
        P = self.P
        xs, hs, ps = {}, {-1: np.copy(h_prev)}, {}
        loss = 0.0
        for t in range(len(inputs)):                         # forward
            xs[t] = one_hot(inputs[t], self.V)
            hs[t] = np.tanh(P["Wxh"] @ xs[t] + P["Whh"] @ hs[t - 1] + P["bh"])
            y = P["Why"] @ hs[t] + P["by"]
            ps[t] = np.exp(y - y.max()) / np.exp(y - y.max()).sum()
            loss += -np.log(ps[t][targets[t], 0])
        G = {k: np.zeros_like(v) for k, v in P.items()}
        dh_next = np.zeros_like(hs[0])
        for t in reversed(range(len(inputs))):                # backward through time
            dy = np.copy(ps[t]); dy[targets[t]] -= 1
            G["Why"] += dy @ hs[t].T; G["by"] += dy
            dh = P["Why"].T @ dy + dh_next                    # from this step's output + from the future
            draw = (1 - hs[t] ** 2) * dh
            G["bh"] += draw; G["Wxh"] += draw @ xs[t].T; G["Whh"] += draw @ hs[t - 1].T
            dh_next = P["Whh"].T @ draw
        for k in G:
            np.clip(G[k], -5, 5, out=G[k])                    # clip exploding gradients
        return loss / len(inputs), G, hs[len(inputs) - 1]

    def update(self, G, lr, b1=0.9, b2=0.999):
        # Adam optimizer (the same update torch.optim.Adam uses)
        self.t += 1
        for k in self.P:
            self.m[k] = b1 * self.m[k] + (1 - b1) * G[k]
            self.v[k] = b2 * self.v[k] + (1 - b2) * G[k] ** 2
            m_hat, v_hat = self.m[k] / (1 - b1 ** self.t), self.v[k] / (1 - b2 ** self.t)
            self.P[k] -= lr * m_hat / (np.sqrt(v_hat) + 1e-8)

### 6c. Sampling with temperature

To generate text we turn the output scores into probabilities and **sample** the next character. *Temperature* $\tau$ controls how adventurous the model is:

$$p_i = \frac{\exp(y_i / \tau)}{\sum_j \exp(y_j / \tau)}$$

Low $\tau$ (e.g. 0.3) is safe and repetitive; high $\tau$ (e.g. 1.5) is creative and messy.

In [ ]:
def sample(model, seed_char, n, temperature=1.0, h=None, rng=None):
    rng = rng or np.random.default_rng()
    P = model.P
    h = np.zeros((model.H, 1)) if h is None else h
    ix = char_to_ix[seed_char]
    out = [seed_char]
    for _ in range(n):
        x = one_hot(ix, model.V)
        h = np.tanh(P["Wxh"] @ x + P["Whh"] @ h + P["bh"])
        y = P["Why"] @ h + P["by"]
        # ✏️ TODO: turn scores y into probabilities p using the temperature, then sample ix from p
        # ✏️ YOUR CODE HERE (hint: divide by temperature, softmax, rng.choice)
        raise NotImplementedError("Complete this TODO")
        out.append(ix_to_char[ix])                 # feed the sampled character back in next step
    return "".join(out)

untrained = CharRNN(V)
print("Before training:\n", sample(untrained, "A", 120, rng=np.random.default_rng(0)))

### 6d. Train and watch it learn

Training takes about a minute. Every 1000 iterations we print a sample; watch it go from noise, to letter patterns, to word-like strings.

In [ ]:
def train_char_rnn(model, data, n_iters=6000, seq_len=25, lr=0.002, print_every=1000, seed=0):
    r = np.random.default_rng(seed)
    pos, h = 0, np.zeros((model.H, 1))
    smooth_loss, losses = np.log(model.V), []
    for it in range(n_iters + 1):
        if pos + seq_len + 1 >= len(data):
            pos, h = 0, np.zeros((model.H, 1))           # back to the start of the text
        inputs = data[pos: pos + seq_len]
        targets = data[pos + 1: pos + seq_len + 1]          # the target is the NEXT character
        loss, G, h = model.loss_and_grads(inputs, targets, h)
        model.update(G, lr)
        smooth_loss = 0.99 * smooth_loss + 0.01 * loss
        losses.append(smooth_loss)
        if print_every and it % print_every == 0:
            print(f"--- iter {it}, loss {smooth_loss:.2f} ---")
            print(sample(model, "T", 150, temperature=0.8, rng=r), "\n")
        pos += seq_len
    return losses

model = CharRNN(V, H=100)
losses = train_char_rnn(model, data)
plt.figure(figsize=(7, 3)); plt.plot(losses); plt.xlabel("iteration"); plt.ylabel("smoothed loss"); plt.title("Char-RNN training"); plt.grid(alpha=.3); plt.show()

In [ ]:
# Explore temperature
for tau in [0.3, 0.8, 1.5]:
    print(f"=== temperature {tau} ===")
    print(sample(model, "A", 200, temperature=tau, rng=np.random.default_rng(1)), "\n")

> 🧠 **Think:**
> 1. Which words did the model learn to spell first? Why might short, frequent words come first?
> 2. Describe how the output changes from τ = 0.3 to τ = 1.5.
> 3. Our text is only ~1,700 characters. Is the model *learning English* or *memorizing this passage*? How could you test that?

---
## Part 7 · *(Colab / PyTorch)* RNN vs. LSTM on the memory task (20 min)

In practice you won't write BPTT by hand: PyTorch's autograd does it for you. This part needs **PyTorch**, which is preinstalled on Google Colab (`Runtime → Change runtime type → GPU` is optional; CPU is fine).

We'll redo the Part 5 experiment with `nn.RNN` and then swap in `nn.LSTM`. **Changing one word** lets us compare a vanilla RNN with a gated one.

In [ ]:
try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
    torch.manual_seed(345)
    print("PyTorch", torch.__version__)
except ImportError:
    HAS_TORCH = False
    print("PyTorch is not installed here. Run Part 7 on Google Colab (or `pip install torch`).")

In [ ]:
assert HAS_TORCH, "Part 7 needs PyTorch. Open this notebook in Google Colab, where it is preinstalled."

class SeqClassifier(nn.Module):
    # Many-to-one: read the whole sequence, classify from the last hidden state.
    def __init__(self, cell="rnn", hidden_size=32):
        super().__init__()
        rnn_cls = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[cell]
        self.rnn = rnn_cls(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 2)

    def forward(self, x):            # x: (batch, T, 1)
        out, _ = self.rnn(x)         # out: (batch, T, hidden), one hidden state per step
        return self.head(out[:, -1]) # use only the LAST hidden state

def torch_memory_batch(B, T):
    x = torch.randint(0, 2, (B, T, 1)).float()
    y = x[:, 0, 0].long()            # label = first bit
    return x, y

def train_torch(cell, T, steps=1500, lr=1e-2):
    model = SeqClassifier(cell)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for step in range(steps):
        x, y = torch_memory_batch(64, T)
        loss = loss_fn(model(x), y)
        opt.zero_grad()
        loss.backward()                                          # <- BPTT, done by autograd
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)        # gradient clipping
        opt.step()
    with torch.no_grad():
        x, y = torch_memory_batch(2000, T)
        return (model(x).argmax(1) == y).float().mean().item()

if HAS_TORCH:
    for T in [10, 40, 100]:
        accs = {cell: train_torch(cell, T) for cell in ["rnn", "lstm"]}
        print(f"T = {T:>3}:  RNN {accs['rnn']:.1%}   LSTM {accs['lstm']:.1%}")

> 🧠 **Think:** At which length does the vanilla RNN fall to chance? Does the LSTM survive? Add `"gru"` to the list. How does it compare? (We'll see *why* LSTMs work in the next lecture.)

### 7b. Char-level generation in PyTorch

The same model as Part 6, in about 25 lines. Notice what you no longer have to write: one-hot vectors (`nn.Embedding` handles it), the backward pass, and the optimizer.

In [ ]:
assert HAS_TORCH, "Part 7 needs PyTorch. Open this notebook in Google Colab, where it is preinstalled."

class TorchCharRNN(nn.Module):
    def __init__(self, V, H=128, cell="rnn"):
        super().__init__()
        self.embed = nn.Embedding(V, H)
        self.rnn = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[cell](H, H, batch_first=True)
        self.head = nn.Linear(H, V)
    def forward(self, ix, state=None):          # ix: (B, T) character indices
        out, state = self.rnn(self.embed(ix), state)
        return self.head(out), state           # (B, T, V) scores for the NEXT character

def torch_sample(model, seed_char, n=200, temperature=0.8):
    ix = torch.tensor([[char_to_ix[seed_char]]]); state, out = None, [seed_char]
    with torch.no_grad():
        for _ in range(n):
            logits, state = model(ix, state)
            p = torch.softmax(logits[0, -1] / temperature, dim=0)
            ix = torch.multinomial(p, 1).view(1, 1)          # feed the sample back in
            out.append(ix_to_char[ix.item()])
    return "".join(out)

if HAS_TORCH:
    data_t = torch.tensor(data)
    cmodel = TorchCharRNN(V, cell="rnn")
    opt = torch.optim.Adam(cmodel.parameters(), lr=3e-3)
    seq_len, B = 50, 32
    for it in range(1501):
        starts = torch.randint(0, len(data_t) - seq_len - 1, (B,))
        xb = torch.stack([data_t[s: s + seq_len] for s in starts])
        yb = torch.stack([data_t[s + 1: s + seq_len + 1] for s in starts])
        logits, _ = cmodel(xb)
        loss = nn.functional.cross_entropy(logits.reshape(-1, V), yb.reshape(-1))
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(cmodel.parameters(), 1.0); opt.step()
        if it % 500 == 0:
            print(f"--- iter {it}, loss {loss.item():.2f} ---\n{torch_sample(cmodel, 'T', 150)}\n")

---
## Part 8 · Practice challenges

Pick at least **two**. Each one builds directly on code you wrote today. ⭐ = warm-up, ⭐⭐ = solid practice, ⭐⭐⭐ = mini-project.

**⭐ 1. Hidden-size sweep.** Train the Part 6 char-RNN with `H = 25, 50, 100, 200`. Plot the final smoothed loss against `H`. Where do you hit diminishing returns? How long does each run take?

**⭐ 2. Truncated BPTT.** In `train_char_rnn`, try `seq_len = 5, 25, 100`. Shorter chunks mean gradients only flow back a few steps. How does that affect the loss and the quality of the samples? Relate it to long-term dependencies.

**⭐⭐ 3. A different memory task.** Change `make_memory_batch` so the label is the **parity (XOR) of all the bits**, or **whether the sequence contains more 1s than 0s**. Which one is harder for the vanilla RNN, and why?

**⭐⭐ 4. Many-to-many tagging.** Build a toy "named entity" dataset: sequences of word IDs where some IDs are "names" (label 1) and the rest are "other" (label 0). Modify `forward_backward` to put a loss on **every** time step (hint: copy the pattern from `CharRNN.loss_and_grads`). Then make it harder: label a word 1 only if the *previous* word was a specific trigger word, like "Dr.". Now the model needs memory.

**⭐⭐ 5. Your own corpus.** Replace `text` with something with a strong pattern: Python source code, song chords, a sonnet (pre-1929 poetry is public domain), or your own class notes. On Colab you can download Tiny Shakespeare:
```python
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
text = open("input.txt").read()[:100_000]
```
What structure does the model pick up (line breaks, indentation, capitalization)?

**⭐⭐⭐ 6. Implement gradient clipping by norm.** Replace the element-wise `np.clip` with global-norm clipping: if $\|g\| > c$, rescale every gradient by $c / \|g\|$. Set `Whh` to a large initial scale so gradients explode, and compare training with no clipping, element-wise clipping, and norm clipping.

**⭐⭐⭐ 7. Sentiment classifier in PyTorch.** Use `SeqClassifier` with an `nn.Embedding` front end to classify short movie reviews as positive or negative (e.g., a small slice of the IMDB dataset from `torchtext` or Hugging Face `datasets`). Compare RNN, GRU and LSTM accuracy, and report how sentence length affects each.

---
### Wrap-up: submit
1. Your completed notebook (all ✅ checks passing).
2. Answers to the 🧠 **Think** questions.
3. One challenge: a short paragraph describing what you tried, one plot, and what you learned.